<a href="https://colab.research.google.com/github/rominaarndt/Adquisici-n-y-Normalizaci-n-con-Pandas/blob/main/ingenier_a_de_features_con_sql_anal_tico_avanzado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ingeniería de Features con SQL Analítico Avanzado

In [ ]:
import sqlite3
import pandas as pd


# 1. Conectar a una base de datos SQLite en memoria

In [ ]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

Creacion de las tablas usuario y pedidos

In [ ]:
# 2. Crear las tablas (adaptando VARCHAR a TEXT y DECIMAL a REAL)

cursor.execute("""
CREATE TABLE IF NOT EXISTS usuarios (
    usuario_id INT,
    nombre VARCHAR
);
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS pedidos (
    pedido_id INT,
    usuario_id INT,
    fecha DATE,
    monto DECIMAL (10,2)
);
""")

In [ ]:
# 3. Insertar los datos en tabla de usuarios
# Limpiamos primero para evitar duplicados en ejecuciones consecutivas
cursor.execute("DELETE FROM usuarios;")

cursor.executemany("INSERT INTO usuarios VALUES (?, ?);", [
    (1, 'Lorena'),
    (2, 'Romina'),
    (3, 'Carlos'),
    (4, 'Dominic')
])

In [ ]:
# Insertar los datos en tabla de PEDIDOS
# Limpiamos primero para evitar duplicados en ejecuciones consecutivas
cursor.execute("DELETE FROM pedidos;")

cursor.executemany("INSERT INTO pedidos VALUES (?, ?, ?, ?);", [
    (1, 1, '2024-01-10', 100),
    (2, 1, '2024-02-15', 150),
    (3, 1, '2024-03-20', 300),
    (4, 2, '2024-01-05', 200),
    (5, 2, '2024-01-25', 100),
    (6, 3, '2024-03-05', 50),
    (7, 3, '2024-03-28', 70)
])
conn.commit()

Features RFM por usuario

In [ ]:
query = """
SELECT
    u.nombre AS usuario,
    CAST((julianday('2024-03-31') - julianday(MAX(p.fecha))) AS INTEGER) AS recency,
    COUNT(DISTINCT p.pedido_id) AS frequency,
    COALESCE(SUM(p.monto), 0) AS monetary
FROM usuarios u
LEFT JOIN pedidos p ON u.usuario_id = p.usuario_id
GROUP BY u.usuario_id, u.nombre;
"""

df_resultado = pd.read_sql(query, conn)
display(df_resultado)

,usuario,recency,frequency,monetary
0,Lorena,11.0,3,550
1,Romina,66.0,2,300
2,Carlos,3.0,2,120
3,Dominic,NaN,0,0


Feature de tendencia (Trend)

In [ ]:
query_trend = """
WITH mensual AS (
  SELECT usuario_id, strftime('%Y-%m-01', fecha) AS mes, SUM(monto) AS gasto
  FROM pedidos GROUP BY usuario_id, strftime('%Y-%m-01', fecha)
),
hist AS (
  SELECT DISTINCT usuario_id,
         AVG(gasto) OVER (PARTITION BY usuario_id) AS prom_hist
  FROM mensual
  WHERE mes < '2024-03-01'          -- excluye el último mes
),
ultimo AS (
  SELECT usuario_id, gasto FROM mensual WHERE mes = '2024-03-01'
)
SELECT u.usuario_id, u.nombre,
       COALESCE(l.gasto, 0)                            AS gasto_ultimo_mes,
       COALESCE(h.prom_hist, 0)                        AS prom_hist,
       COALESCE(l.gasto, 0) - COALESCE(h.prom_hist, 0) AS trend
FROM usuarios u
LEFT JOIN hist   h ON u.usuario_id = h.usuario_id
LEFT JOIN ultimo l ON u.usuario_id = l.usuario_id;
"""

df_trend = pd.read_sql(query_trend, conn)
display(df_trend)

,usuario_id,nombre,gasto_ultimo_mes,prom_hist,trend
0,1,Lorena,300,125.0,175.0
1,2,Romina,0,300.0,-300.0
2,3,Carlos,120,0.0,120.0
3,4,Dominic,0,0.0,0.0


```markdown
### Consolidación Final: RFM, Trend y Reglas de Negocio (Churn)
Combinamos ambas métricas en una única consulta para construir el dataset de entrenamiento o análisis final.
```

In [ ]:
query_final = """
WITH rfm AS (
    SELECT
        u.usuario_id,
        u.nombre,
        CAST((julianday('2024-03-31') - julianday(MAX(p.fecha))) AS INTEGER) AS recency,
        COUNT(DISTINCT p.pedido_id) AS frequency,
        COALESCE(SUM(p.monto), 0) AS monetary
    FROM usuarios u
    LEFT JOIN pedidos p ON u.usuario_id = p.usuario_id
    GROUP BY u.usuario_id, u.nombre
),
mensual AS (
    SELECT usuario_id, strftime('%Y-%m-01', fecha) AS mes, SUM(monto) AS gasto
    FROM pedidos GROUP BY usuario_id, strftime('%Y-%m-01', fecha)
),
hist AS (
    SELECT DISTINCT usuario_id,
           AVG(gasto) OVER (PARTITION BY usuario_id) AS prom_hist
    FROM mensual
    WHERE mes < '2024-03-01'
),
ultimo AS (
    SELECT usuario_id, gasto FROM mensual WHERE mes = '2024-03-01'
),
tend AS (
    SELECT u.usuario_id,
           COALESCE(l.gasto, 0) - COALESCE(h.prom_hist, 0) AS trend
    FROM usuarios u
    LEFT JOIN hist   h ON u.usuario_id = h.usuario_id
    LEFT JOIN ultimo l ON u.usuario_id = l.usuario_id
)
SELECT
    r.usuario_id,
    r.nombre,
    COALESCE(r.recency, 999) AS recency,
    r.frequency,
    r.monetary,
    t.trend,
    CASE
        WHEN r.frequency = 0 OR COALESCE(r.recency, 999) > 60 THEN 1
        ELSE 0
    END AS churn
FROM rfm r
JOIN tend t ON r.usuario_id = t.usuario_id;
"""

df_final = pd.read_sql(query_final, conn)
display(df_final)

,usuario_id,nombre,recency,frequency,monetary,trend,churn
0,1,Lorena,11,3,550,175.0,0
1,2,Romina,66,2,300,-300.0,1
2,3,Carlos,3,2,120,120.0,0
3,4,Dominic,999,0,0,0.0,1


```markdown
### Generación del Archivo de Documentación (README.md)
Ejecuta la siguiente celda para crear un archivo `README.md` que documente formalmente la metodología, el esquema de datos y las métricas implementadas en este proyecto analítico.
```

In [ ]:
readme_content = """# Ingeniería de Features con SQL Analítico Avanzado

Este proyecto implementa un modelo analítico de segmentación y predicción de comportamiento de usuarios utilizando SQL avanzado (SQLite) y Python/Pandas. A partir de los datos transaccionales de compras, se calculan variables de comportamiento clave para cada usuario.

## 📋 Estructura de Datos
El modelo utiliza dos tablas principales creadas en una base de datos SQLite en memoria:
1. **usuarios** (`usuario_id`, `nombre`)
2. **pedidos** (`pedido_id`, `usuario_id`, `fecha`, `monto`)

## 📊 Métricas Calculadas
Mediante una consulta consolidada con expresiones de tabla común (`WITH ... AS`), se derivan las siguientes variables analíticas:

* **RFM (Recency, Frequency, Monetary):**
  * **Recency (Recencia):**
    * **Qué mide:** Los días transcurridos desde la última compra del usuario hasta la fecha de referencia (corte en `2024-03-31`).
    * **Cómo se calcula:** `julianday('2024-03-31') - julianday(MAX(fecha))` por usuario (equivalente a `DATE_DIFF` de días).
    * **Nulos:** Los usuarios sin compras reciben por defecto un valor de `999` (indica que nunca han comprado).
    * **Por qué sirve para churn:** A mayor recencia, existe una mayor probabilidad estadística de abandono o inactividad prolongada.
  * **Frequency (Frecuencia):** Número total de pedidos únicos realizados por el usuario.
  * **Monetary (Valor Monetario):** Gasto total acumulado por el usuario.

* **Trend (Tendencia de Gasto):**
  * Mide el cambio en el comportamiento financiero del cliente comparando el gasto realizado en el último mes activo (`2024-03-01`) contra el promedio de gasto mensual de los meses históricos anteriores.

* **Churn (Fuga de Clientes):**
  * Variable binaria predictiva (`1` o `0`). Se clasifica a un usuario en situación de Churn (`1`) si cumple con alguna de las siguientes condiciones de negocio:
    1. No ha realizado ninguna transacción en el historial (`frequency = 0`).
    2. Su última transacción fue hace más de 60 días (`recency > 60`).

## 🛠️ Requisitos de Ejecución
* Python 3.x
* Biblioteca `pandas`
* Biblioteca estándar `sqlite3`
"""

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("¡Archivo README.md actualizado con éxito incorporando el detalle de Recency!")

¡Archivo README.md actualizado con éxito incorporando el detalle de Recency!


```markdown
### Visualización del archivo README.md generado
Ejecuta la siguiente celda para leer y mostrar el contenido completo del archivo de documentación en formato Markdown.
```

In [ ]:
from IPython.display import Markdown

# Leer el contenido del archivo
with open("README.md", "r", encoding="utf-8") as f:
    contenido = f.read()

# Mostrar el Markdown renderizado
display(Markdown(contenido))

# Ingeniería de Features con SQL Analítico Avanzado

Este proyecto implementa un modelo analítico de segmentación y predicción de comportamiento de usuarios utilizando SQL avanzado (SQLite) y Python/Pandas. A partir de los datos transaccionales de compras, se calculan variables de comportamiento clave para cada usuario.

## 📋 Estructura de Datos
El modelo utiliza dos tablas principales creadas en una base de datos SQLite en memoria:
1. **usuarios** (`usuario_id`, `nombre`)
2. **pedidos** (`pedido_id`, `usuario_id`, `fecha`, `monto`)

## 📊 Métricas Calculadas
Mediante una consulta consolidada con expresiones de tabla común (`WITH ... AS`), se derivan las siguientes variables analíticas:

* **RFM (Recency, Frequency, Monetary):**
  * **Recency (Recencia):** 
    * **Qué mide:** Los días transcurridos desde la última compra del usuario hasta la fecha de referencia (corte en `2024-03-31`).
    * **Cómo se calcula:** `julianday('2024-03-31') - julianday(MAX(fecha))` por usuario (equivalente a `DATE_DIFF` de días).
    * **Nulos:** Los usuarios sin compras reciben por defecto un valor de `999` (indica que nunca han comprado).
    * **Por qué sirve para churn:** A mayor recencia, existe una mayor probabilidad estadística de abandono o inactividad prolongada.
  * **Frequency (Frecuencia):** Número total de pedidos únicos realizados por el usuario.
  * **Monetary (Valor Monetario):** Gasto total acumulado por el usuario.

* **Trend (Tendencia de Gasto):**
  * Mide el cambio en el comportamiento financiero del cliente comparando el gasto realizado en el último mes activo (`2024-03-01`) contra el promedio de gasto mensual de los meses históricos anteriores.

* **Churn (Fuga de Clientes):**
  * Variable binaria predictiva (`1` o `0`). Se clasifica a un usuario en situación de Churn (`1`) si cumple con alguna de las siguientes condiciones de negocio:
    1. No ha realizado ninguna transacción en el historial (`frequency = 0`).
    2. Su última transacción fue hace más de 60 días (`recency > 60`).

## 🛠️ Requisitos de Ejecución
* Python 3.x
* Biblioteca `pandas`
* Biblioteca estándar `sqlite3`
